In [1]:
"""
异常值检测优化方案 v2.0

主要改进：
1. 第一层物理约束: 为所有变量添加合理的物理边界约束
2. 第二层统计检测: 使用特征-目标相关性分析
3. 第三层聚类检测: 整合PCA异常检测流程
4. 参数合理性验证: 检查K值和Z阈值设置
5. 模块化设计: 每个部分功能单一明确
"""

import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('依赖库导入完成')

# ==============================================================================
# 数据加载和预处理
# ==============================================================================
print('\n' + '='*60)
print('数据加载和预处理')
print('='*60)

# 加载数据
try:
    df = pd.read_excel('df_imputed.xlsx')
    print(f'数据形状: {df.shape}')
except FileNotFoundError:
    print("错误: 找不到文件 'df_imputed.xlsx'")
    print("请确保数据文件存在于当前目录")
    exit(1)
except Exception as e:
    print(f"加载数据时发生错误: {e}")
    exit(1)

FEATURES = ['Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Zn', 'V', 'Ti', 'Zr', 'Li', 'Ni', 'Be', 'Sc', 'Tsol', 'Tage', 'tage']
TARGETS = ['YS', 'UTS', 'El']
ALL_VARS = FEATURES + TARGETS

# 验证数据列是否存在
missing_cols = [col for col in ALL_VARS if col not in df.columns]
if missing_cols:
    print(f"错误: 数据中缺少以下列: {missing_cols}")
    print(f"数据中的列: {list(df.columns)}")
    exit(1)

print(f'特征变量: {len(FEATURES)}个')
print(f'目标变量: {len(TARGETS)}个')
print(f'总变量数: {len(ALL_VARS)}个')

# ==============================================================================
# 第一层：全面的物理边界约束
# ==============================================================================
print('\n' + '='*60)
print('第一层：物理边界检查')
print('='*60)

# 物理边界约束
PHYSICAL_BOUNDS = {
    # 化学成分 (wt%) - 基于锻造铝合金特性优化
    'Si': (0, 20),      # 覆盖所有铝合金系列
    'Fe': (0, 5),       # 宽松边界
    'Cu': (0, 10),      # 覆盖2xxx/6xxx/7xxx系列
    'Mn': (0, 2),       # 常规范围
    'Mg': (0, 5),       # 常规范围
    'Cr': (0, 2),       # 微量添加元素
    'Zn': (0, 8),       # 7xxx系列主要合金元素
    'V': (0, 1),        # 微量元素
    'Ti': (0, 1),       # 晶粒细化元素
    'Zr': (0, 1),       # 微量元素
    'Li': (0, 0.5),     # 【关键最优参数】
    'Ni': (0, 2),       # 微量元素
    'Be': (0, 0.1),     # 极微量元素
    'Sc': (0, 0.5),     # 微量元素

    # 工艺参数 - 覆盖所有真实热处理工艺
    'Tsol': (350, 620), # 固溶温度：低温到高温
    'Tage': (80, 350),  # 时效温度：自然时效到人工时效
    'tage': (0.1, 72),  # 时效时间：短时到长时

    # 力学性能 - 覆盖典型锻造铝合金性能范围
    'YS': (80, 850),    # 屈服强度
    'UTS': (120, 950),  # 抗拉强度
    'El': (0, 70)       # 延伸率
}
    

physical_outliers = set()
print('物理边界约束检查:')

for col in ALL_VARS:
    if col in PHYSICAL_BOUNDS:
        data = df[col].dropna()
        lo, hi = PHYSICAL_BOUNDS[col]

        # 检查边界合理性
        if lo >= hi:
            print(f'  {col}: 边界设置不合理 ({lo} >= {hi})，跳过')
            continue

        mask = (data < lo) | (data > hi)
        outlier_count = mask.sum()

        if outlier_count > 0:
            outlier_indices = data[mask].index.tolist()
            physical_outliers.update(outlier_indices)
            print(f'  {col}: {outlier_count}个异常值 (范围: [{lo}, {hi}])')
        else:
            print(f'  {col}: 无异常值 (范围: [{lo}, {hi}])')
    else:
        print(f'  {col}: 未定义物理边界')

print(f'\n物理边界异常总计: {len(physical_outliers)}个样本')

# ==============================================================================
# 第二层：基于特征-目标相关性的异常检测
# ==============================================================================
print('\n' + '='*60)
print('第二层：特征-目标相关性异常检测')
print('='*60)

statistical_outliers = set()
correlation_outliers = set()
print('基于相关性的异常检测:')

for target in TARGETS:
    print(f'\n分析目标变量: {target}')

    # 计算所有特征与目标的相关性
    correlations = {}
    for feature in FEATURES:
        valid_data = df[[feature, target]].dropna()
        if len(valid_data) > 10:
            corr, p_val = stats.pearsonr(valid_data[feature], valid_data[target])
            correlations[feature] = {'corr': corr, 'p_val': p_val}

    # 找出显著相关的特征 (p < 0.05 且 |corr| > 0.3)
    significant_features = [f for f, stats_dict in correlations.items()
                           if stats_dict['p_val'] < 0.05 and abs(stats_dict['corr']) > 0.3]

    print(f'  显著相关特征: {len(significant_features)}个')

    if significant_features:
        # 对显著相关特征进行残差分析
        X = df[significant_features].dropna()
        y = df.loc[X.index, target]

        if len(X) > 10:
            # 使用随机森林计算预测值和残差
            rf = RandomForestRegressor(random_state=42)
            rf.fit(X, y)
            predictions = rf.predict(X)
            residuals = np.abs(y - predictions)

            # 使用IQR方法检测残差异常值
            Q1, Q3 = np.percentile(residuals, [25, 75])
            IQR = Q3 - Q1
            residual_threshold = Q3 + 3 * IQR

            residual_outliers = residuals[residuals > residual_threshold].index.tolist()
            correlation_outliers.update(residual_outliers)

            print(f'  {target}: 检测到{len(residual_outliers)}个相关性异常值')

statistical_outliers = correlation_outliers
print(f'\n相关性异常总计: {len(statistical_outliers)}个样本')

# ==============================================================================
# 第三层：PCA聚类异常检测
# ==============================================================================
print('\n' + '='*60)
print('第三层：PCA聚类异常检测')
print('='*60)

# 数据预处理
df_pca = df[FEATURES].dropna().copy()
X_scaled = StandardScaler().fit_transform(df_pca)
print(f'PCA分析样本数: {len(df_pca)}')

# PCA降维
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA保留主成分数: {pca.n_components_}')
print(f'解释方差比例: {np.sum(pca.explained_variance_ratio_):.3f}')

# ==============================================================================
# K值合理性验证
# ==============================================================================
print('\n聚类数K选择分析:')

# 设置更合理的K值范围
max_k = min(15, int(len(df_pca) * 0.15))  # 最多15个聚类或15%样本数
k_range = range(2, max_k + 1)
print(f'K值搜索范围: 2-{max_k} (基于样本量{len(df_pca)}的约束)')

sil_scores = []
inertias = []
k_values = []

for k in k_range:
    if len(df_pca) < k * 5:  # 每个聚类至少5个样本
        print(f'  K={k}: 样本数不足，跳过')
        continue

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)

    sil_score = silhouette_score(X_pca, labels)
    inertia = km.inertia_

    sil_scores.append(sil_score)
    inertias.append(inertia)
    k_values.append(k)

    print(f'  K={k}: 轮廓系数={sil_score:.3f}, 惯性={inertia:.1f}')

# 选择最优K值
if sil_scores:
    optimal_k_sil = k_values[np.argmax(sil_scores)]
    print(f'\n基于轮廓系数最优K值: {optimal_k_sil}')

    # 检查K值合理性
    max_reasonable_k = int(len(df_pca) * 0.1)  # 最多10%的聚类数
    if optimal_k_sil > max_reasonable_k:
        optimal_k = max(2, int(len(df_pca) * 0.05))  # 调整为5%
        print(f'K值{optimal_k_sil}过大(>{max_reasonable_k})，调整为: {optimal_k}')
    else:
        optimal_k = optimal_k_sil
        print(f'K值{optimal_k}合理 (≤{max_reasonable_k})')
else:
    optimal_k = 3
    print(f'\n无法确定最优K值，使用默认值: {optimal_k}')

# ==============================================================================
# 执行聚类和距离计算
# ==============================================================================
print(f'\n执行K-means聚类 (K={optimal_k})')
km = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
labels = km.fit_predict(X_pca)
centers = km.cluster_centers_

# 计算样本到聚类中心的距离
dists = np.array([np.linalg.norm(X_pca[i] - centers[labels[i]])
                  for i in range(len(X_pca))])

print(f'距离统计: 最小值={dists.min():.3f}, 最大值={dists.max():.3f}')
print(f'距离统计: Q1={np.percentile(dists, 25):.3f}, 中位数={np.median(dists):.3f}, Q3={np.percentile(dists, 75):.3f}')

# 检查Z阈值合理性
print('\nZ-score阈值分析:')
cluster_stats = {}
for c in range(optimal_k):
    mask = labels == c
    cluster_size = mask.sum()

    if cluster_size >= 5:
        cluster_dists = dists[mask]
        mu, sigma = cluster_dists.mean(), cluster_dists.std()

        # 检查sigma是否为0或过小
        if sigma < 1e-10 or np.isnan(sigma):
            sigma = max(np.mean(dists) * 0.1, 1e-8)

        cluster_stats[c] = {'mu': mu, 'sigma': sigma, 'size': cluster_size, 'mask': mask}

        print(f'  聚类{c}: 样本数={cluster_size}, μ={mu:.3f}, σ={sigma:.3f}')
    else:
        print(f'  聚类{c}: 样本数过少({cluster_size})，跳过')
        cluster_stats[c] = {'mu': 0, 'sigma': 1, 'size': cluster_size, 'mask': mask}

# ==============================================================================
# Z-score阈值优化和异常检测
# ==============================================================================
print('\n' + '='*60)
print('Z-score阈值优化')
print('='*60)

# 测试不同的Z阈值
z_thresholds = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
threshold_results = []

print('Z阈值合理性分析:')
for z_thresh in z_thresholds:
    pca_outliers_current = set()

    for c in range(optimal_k):
        stats_c = cluster_stats[c]
        if stats_c['size'] >= 5 and stats_c['sigma'] > 0:
            mask = stats_c['mask']
            z_scores = (dists[mask] - stats_c['mu']) / stats_c['sigma']
            outlier_mask = z_scores >= z_thresh

            if outlier_mask.sum() > 0:
                outlier_indices = df_pca.index[mask][outlier_mask].tolist()
                pca_outliers_current.update(outlier_indices)

    outlier_ratio = len(pca_outliers_current) / len(df)

    # 判断合理性
    if outlier_ratio < 0.01:
        reasonableness = "过低"
    elif outlier_ratio > 0.20:
        reasonableness = "过高"
    elif 0.02 <= outlier_ratio <= 0.15:
        reasonableness = "合理"
        threshold_results.append((z_thresh, len(pca_outliers_current), outlier_ratio))
    else:
        reasonableness = "边缘"

    print(f'Z={z_thresh}: 异常值{len(pca_outliers_current)}个, 剔除比例={outlier_ratio:.1%} ({reasonableness})')

# 选择最优阈值
print('\nZ阈值选择逻辑:')
if threshold_results:
    # 优先选择剔除比例在5-8%之间的阈值
    preferred_results = [r for r in threshold_results if 0.05 <= r[2] <= 0.08]

    if preferred_results:
        best_threshold = min(preferred_results, key=lambda x: abs(x[2] - 0.065))[0]
        print(f'选择5-8%范围内的最优Z阈值: {best_threshold} (剔除比例: {min(preferred_results, key=lambda x: abs(x[2] - 0.065))[2]:.1%})')
    else:
        best_threshold = min(threshold_results, key=lambda x: abs(x[2] - 0.05))[0]
        print(f'选择最接近5%的Z阈值: {best_threshold} (剔除比例: {min(threshold_results, key=lambda x: abs(x[2] - 0.05))[2]:.1%})')
else:
    # 如果没有合理范围内的结果，选择剔除比例最接近5%的
    all_results = []
    for z in z_thresholds:
        try:
            outlier_count = sum([stats_c["size"] for c in range(optimal_k)
                               if cluster_stats[c]['size'] >= 5 and cluster_stats[c]['sigma'] > 0
                               and np.any((dists[cluster_stats[c]['mask']] - cluster_stats[c]['mu']) / cluster_stats[c]['sigma'] >= z)])
            outlier_ratio = outlier_count / len(df)
            all_results.append((z, outlier_ratio))
        except:
            all_results.append((z, 0))

    if all_results:
        best_threshold = min(all_results, key=lambda x: abs(x[1] - 0.05))[0]
        print(f'无合理范围内的Z阈值，选择最接近5%的: {best_threshold}')
    else:
        best_threshold = 2.0
        print(f'无法计算Z阈值，使用默认值: {best_threshold}')

# 使用最优阈值进行最终异常检测
pca_outliers = set()
for c in range(optimal_k):
    stats_c = cluster_stats[c]
    if stats_c['size'] >= 5 and stats_c['sigma'] > 0:
        mask = stats_c['mask']
        z_scores = (dists[mask] - stats_c['mu']) / stats_c['sigma']
        outlier_mask = z_scores >= best_threshold

        if outlier_mask.sum() > 0:
            outlier_indices = df_pca.index[mask][outlier_mask].tolist()
            pca_outliers.update(outlier_indices)
            print(f'聚类{c}: {outlier_mask.sum()}个异常值')

print(f'\nPCA聚类异常总计: {len(pca_outliers)}个样本 (Z={best_threshold})')

# ==============================================================================
# 异常值汇总
# ==============================================================================
print('\n' + '='*60)
print('异常值检测汇总')
print('='*60)

# 汇总所有异常值
all_outliers = physical_outliers | statistical_outliers | pca_outliers

print('各层异常值统计:')
print(f'  第一层(物理边界): {len(physical_outliers)}个')
print(f'  第二层(相关性): {len(statistical_outliers)}个')
print(f'  第三层(PCA聚类): {len(pca_outliers)}个')
print(f'  总计异常样本: {len(all_outliers)}个')
print(f'  剔除比例: {len(all_outliers)/len(df)*100:.1f}%')

# 检查是否有重复检测
overlap_physical_stat = len(physical_outliers & statistical_outliers)
overlap_physical_pca = len(physical_outliers & pca_outliers)
overlap_stat_pca = len(statistical_outliers & pca_outliers)

print(f'\n异常值重叠情况:')
print(f'  物理∩统计: {overlap_physical_stat}个')
print(f'  物理∩PCA: {overlap_physical_pca}个')
print(f'  统计∩PCA: {overlap_stat_pca}个')

clean_indices = [i for i in range(len(df)) if i not in all_outliers]
print(f'\n清洗后数据: {len(clean_indices)}个样本')

# ==============================================================================
# 异常值剔除效果验证
# ==============================================================================
print('\n' + '='*60)
print('异常值剔除效果验证')
print('='*60)

for target in TARGETS:
    print(f'\n--- 评估 {target} ---')

    # 准备数据
    X_before, y_before = df[FEATURES].values, df[target].values
    X_after, y_after = df.loc[clean_indices, FEATURES].values, df.loc[clean_indices, target].values

    print(f'  原始样本: {len(X_before)} → 清洗后: {len(X_after)}')

    # 100次重复实验
    metrics_before = {'MAE': [], 'RMSE': [], 'R2': []}
    metrics_after = {'MAE': [], 'RMSE': [], 'R2': []}

    for seed in range(100):
        # 原始数据
        X_tr_bef, X_te_bef, y_tr_bef, y_te_bef = train_test_split(
            X_before, y_before, test_size=0.2, random_state=seed)

        # 清洗后数据
        X_tr_aft, X_te_aft, y_tr_aft, y_te_aft = train_test_split(
            X_after, y_after, test_size=0.2, random_state=seed)

        # 训练和评估
        rf = RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1)

        # 原始数据性能
        rf.fit(X_tr_bef, y_tr_bef)
        pred_bef = rf.predict(X_te_bef)
        metrics_before['MAE'].append(mean_absolute_error(y_te_bef, pred_bef))
        metrics_before['RMSE'].append(np.sqrt(mean_squared_error(y_te_bef, pred_bef)))
        metrics_before['R2'].append(r2_score(y_te_bef, pred_bef))

        # 清洗后数据性能
        rf.fit(X_tr_aft, y_tr_aft)
        pred_aft = rf.predict(X_te_aft)
        metrics_after['MAE'].append(mean_absolute_error(y_te_aft, pred_aft))
        metrics_after['RMSE'].append(np.sqrt(mean_squared_error(y_te_aft, pred_aft)))
        metrics_after['R2'].append(r2_score(y_te_aft, pred_aft))

    # 统计检验和结果分析
    def analyze_improvement(before_list, after_list, metric_name):
        bef_mean, bef_std = np.mean(before_list), np.std(before_list)
        aft_mean, aft_std = np.mean(after_list), np.std(after_list)

        t_stat, p_val = stats.ttest_rel(before_list, after_list)

        # 判断是否改善 (R2越大越好，其他指标越小越好)
        improved = (metric_name == 'R2' and aft_mean > bef_mean) or (metric_name != 'R2' and aft_mean < bef_mean)
        significant = p_val < 0.05

        print(f'  {metric_name}:')
        print(f'    异常值剔除前: {bef_mean:.4f} ± {bef_std:.4f}')
        print(f'    异常值剔除后: {aft_mean:.4f} ± {aft_std:.4f}')
        print(f'    统计检验: t={t_stat:.3f}, p={p_val:.4f}')
        print(f'    结果: {"显著" if significant else "不显著"}{"改善" if improved else "恶化"}')

        return improved and significant

    # 分析各项指标
    mae_significant = analyze_improvement(metrics_before['MAE'], metrics_after['MAE'], 'MAE')
    rmse_significant = analyze_improvement(metrics_before['RMSE'], metrics_after['RMSE'], 'RMSE')
    r2_significant = analyze_improvement(metrics_before['R2'], metrics_after['R2'], 'R2')

    # 综合评估
    improvements = sum([mae_significant, rmse_significant, r2_significant])
    print(f'  综合评估: {improvements}/3 项指标显著改善')

    if improvements >= 2:
        print(f'  [OK] {target}: 异常值剔除效果良好')
    else:
        print(f'  [LIMITED] {target}: 异常值剔除效果有限')

# ==============================================================================
# 最终结论
# ==============================================================================
print('\n' + '='*60)
print('优化方案总结')
print('='*60)

print('\n主要改进:')
print('1. [OK] 第一层: 为所有变量添加了合理的物理边界约束')
print('2. [OK] 第二层: 使用特征-目标相关性分析进行异常检测')
print('3. [OK] 第三层: 整合了完整的PCA聚类异常检测流程')
print('4. [OK] 参数优化: K值和Z阈值都经过合理性验证')
print('5. [OK] 模块化设计: 每个部分功能单一明确')

print('\n参数选择合理性:')
print(f'- 聚类数K={optimal_k}: 基于轮廓系数和样本量约束')
print(f'- Z阈值={best_threshold}: 基于异常值剔除比例优化')
print(f'- 总剔除比例: {len(all_outliers)/len(df)*100:.1f}% (建议范围: 2-15%)')

print('\n建议:')
outlier_ratio = len(all_outliers)/len(df)
if outlier_ratio <= 0.10:
    print('[OK] 异常值剔除比例合理，可以考虑应用此清洗方案')
elif outlier_ratio <= 0.20:
    print('[CAUTION] 异常值剔除比例偏高，建议谨慎评估检测阈值')
else:
    print('[WARNING] 异常值剔除比例过高，建议重新调整检测参数')

print('\n下一步:')
print('1. 根据验证结果决定是否保存清洗后的数据')
print('2. 如果效果不理想，可以调整各层检测参数')
print('3. 考虑使用清洗后的数据进行后续建模')

# 提供保存清洗后数据的选项
clean_df = df.loc[clean_indices].copy()
try:
    clean_df.to_excel('df_cleaned.xlsx', index=False)
    print(f'\n[OK] 清洗后的数据已保存到: df_cleaned.xlsx ({len(clean_df)}个样本)')
except Exception as e:
    print(f'\n[ERROR] 保存清洗后数据失败: {e}')

依赖库导入完成

数据加载和预处理
数据形状: (491, 23)
特征变量: 17个
目标变量: 3个
总变量数: 20个

第一层：物理边界检查
物理边界约束检查:
  Si: 无异常值 (范围: [0, 20])
  Fe: 无异常值 (范围: [0, 5])
  Cu: 无异常值 (范围: [0, 10])
  Mn: 2个异常值 (范围: [0, 2])
  Mg: 无异常值 (范围: [0, 5])
  Cr: 无异常值 (范围: [0, 2])
  Zn: 3个异常值 (范围: [0, 8])
  V: 无异常值 (范围: [0, 1])
  Ti: 无异常值 (范围: [0, 1])
  Zr: 无异常值 (范围: [0, 1])
  Li: 94个异常值 (范围: [0, 0.5])
  Ni: 无异常值 (范围: [0, 2])
  Be: 无异常值 (范围: [0, 0.1])
  Sc: 无异常值 (范围: [0, 0.5])
  Tsol: 无异常值 (范围: [350, 620])
  Tage: 59个异常值 (范围: [80, 350])
  tage: 74个异常值 (范围: [0.1, 72])
  YS: 3个异常值 (范围: [80, 850])
  UTS: 4个异常值 (范围: [120, 950])
  El: 无异常值 (范围: [0, 70])

物理边界异常总计: 158个样本

第二层：特征-目标相关性异常检测
基于相关性的异常检测:

分析目标变量: YS
  显著相关特征: 6个
  YS: 检测到9个相关性异常值

分析目标变量: UTS
  显著相关特征: 6个
  UTS: 检测到4个相关性异常值

分析目标变量: El
  显著相关特征: 2个
  El: 检测到16个相关性异常值

相关性异常总计: 25个样本

第三层：PCA聚类异常检测
PCA分析样本数: 491
PCA保留主成分数: 11
解释方差比例: 0.906

聚类数K选择分析:
K值搜索范围: 2-15 (基于样本量491的约束)
  K=2: 轮廓系数=0.260, 惯性=5935.0
  K=3: 轮廓系数=0.311, 惯性=5044.2
  K=4: 轮廓系数=0.315, 惯性=4538.5
  K=5: 轮廓系数=0.3